# Using the BaseMethod Module in baseobjects

## Introduction

The `BaseMethod` module provides an abstract class that implements the structure for creating method-like callable objects. It extends `BaseCallable` to create callable objects that behave like methods by directly implementing functionality in methods, maintaining a reference to the instance they're bound to and properly handling method binding semantics.

**The primary purpose of BaseMethod is to directly implement functionality in its own methods** through subclassing and overriding `__call__`. This approach provides better performance, more flexibility, and cleaner code when creating custom method-like objects. While BaseMethod can also wrap existing methods, this is a secondary purpose and generally less powerful than direct implementation.

Unlike regular functions, methods are bound to a specific instance and receive that instance as their first argument (typically named 'self'). BaseMethod implements this behavior by storing a weak reference to the bound instance and using it when the method is called. The weak reference prevents memory leaks that could occur if the method held a strong reference to the instance.

> **Note:** For decorator functionality, use the `BaseDecorator` class from the `baseobjects.functions` package instead. `BaseDecorator` is specifically designed for creating decorators with extended functionality.

This tutorial will guide you through:
- Understanding the purpose and design of `BaseMethod`
- Creating custom method-like objects with direct implementation (primary usage)
- Wrapping existing methods (secondary usage)
- Binding methods to instances
- Handling method binding semantics

**Prerequisites:**
- Basic understanding of Python's method binding mechanism
- Familiarity with Python's descriptor protocol
- Knowledge of the `BaseCallable` class from the baseobjects package

### Table of Contents

- [Importing the Module](#Importing-the-Module)
- [Core Functionality](#Core-Functionality)
- [Direct Implementation Examples](#Direct-Implementation-Examples)
- [Secondary Usage - Method Wrapping](#Secondary-Usage---Method-Wrapping)
- [Module Interaction](#Module-Interaction)
- [Advanced Features](#Advanced-Features)
- [Examples](#Examples)
- [API Highlights](#API-Highlights)
- [Troubleshooting / FAQs](#Troubleshooting-/-FAQs)
- [Conclusion and Next Steps](#Conclusion-and-Next-Steps)

## Importing the Module

In [31]:
from baseobjects.bases.basecallable import BaseMethod
import weakref

## Core Functionality

The `BaseMethod` class is an abstract class that extends `BaseCallable` to create callable objects that behave like methods. Its primary purpose is to allow you to directly implement functionality in methods by subclassing and overriding `__call__`, while maintaining a weak reference to the instance they're bound to and properly handling method binding semantics.

### Key Features

1. **Direct Implementation**: Allows you to directly implement functionality by subclassing and overriding `__call__` (primary purpose)
2. **Instance Access**: Provides access to the bound instance through `self.__self__`
3. **Method Binding**: Maintains a reference to the bound instance and properly handles method binding
4. **Weak References**: Uses weak references to prevent memory leaks
5. **Binding Control**: Provides control over binding behavior through the `is_binding` flag
6. **Attribute Binding**: Can bind to instances and set itself as an attribute on the instance

## Direct Implementation Examples

The most powerful way to use BaseMethod is through direct implementation by subclassing and overriding `__call__`. This approach gives you complete control over the method's behavior while still benefiting from the method binding semantics provided by BaseMethod.

Let's create a simple method-like object using `BaseMethod` with direct implementation:

In [32]:
# Create a custom method with direct implementation
class Greeting(BaseMethod):
    """A custom greeting method."""
    
    def __call__(self, name):
        """Generate a greeting for the given name."""
        # Access the bound instance through self.__self__
        if self.__self__ is not None:
            return f"{self.__self__.name} says hello to {name}!"
        else:
            return f"Hello to {name} (unbound greeting)"

# Create a class with a name attribute
class Person:
    def __init__(self, name):
        self.name = name

# Create a Person instance
alice = Person("Alice")

# Create a Greeting instance and bind it to alice
greeting = Greeting(instance=alice, owner=Person)

# Call the method
print(greeting("Bob"))

# Check if the docstring is preserved
print(f"Docstring: {greeting.__call__.__doc__}")

Alice says hello to Bob!
Docstring: Generate a greeting for the given name.


In [33]:
# Let's create another example with more functionality
class Counter(BaseMethod):
    """A method that counts how many times it's been called."""
    
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.count = 0
    
    def __call__(self):
        """Increment the counter and return the new count."""
        self.count += 1
        if self.__self__ is not None:
            return f"{self.__self__.name}'s counter: {self.count}"
        else:
            return f"Unbound counter: {self.count}"
    
    def reset(self):
        """Reset the counter to zero."""
        self.count = 0
        return "Counter reset"

# Create a class that will use our counter
class User:
    def __init__(self, name):
        self.name = name
        # Create a Counter instance and bind it to this User instance
        self.count_visits = Counter(instance=self, owner=User)

# Create a User instance
user = User("Alice")

# Call the counter method multiple times
print(user.count_visits())
print(user.count_visits())
print(user.count_visits())

# Reset the counter
print(user.count_visits.reset())

# Call the counter again
print(user.count_visits())

Alice's counter: 1
Alice's counter: 2
Alice's counter: 3
Counter reset
Alice's counter: 1


## Secondary Usage - Method Wrapping

While the primary purpose of BaseMethod is direct implementation, it can also wrap existing methods. This is a secondary usage that provides a convenient way to enhance existing methods with additional functionality.

### Method Binding with Wrapped Methods

When a `BaseMethod` wraps an existing method and is accessed through an instance (e.g., `instance.method`), it returns itself with the instance bound to it. This mimics Python's standard method binding behavior.

In [34]:
# Define a class with a method
class Greeter:
    def __init__(self, greeting):
        self.greeting = greeting

    def greet(self, name):
        return f"{self.greeting}, {name}!"

# Create a BaseMethod that wraps the greet method
method_greet = BaseMethod(Greeter.greet)

# Create Greeter instances
formal_greeter = Greeter("Good day")
casual_greeter = Greeter("Hey")

# Bind the method to different instances
formal_greet = method_greet.__get__(formal_greeter, Greeter)
casual_greet = method_greet.__get__(casual_greeter, Greeter)

# Call the bound methods
print(formal_greet("Alice"))
print(casual_greet("Bob"))

# Check if the methods are bound to the correct instances
print(f"formal_greet.__self__.greeting: {formal_greet.__self__.greeting}")
print(f"casual_greet.__self__.greeting: {casual_greet.__self__.greeting}")

Hey, Alice!
Hey, Bob!
formal_greet.__self__.greeting: Hey
casual_greet.__self__.greeting: Hey


### More Direct Implementation Examples

Let's explore more examples of direct implementation with BaseMethod, showing how to use binding control and other advanced features.

#### Direct Implementation with Binding Control

You can control whether a BaseMethod binds to new instances by setting the `is_binding` flag. This is useful for creating methods that behave like static methods.

In [35]:
# Create a custom method with direct implementation and binding control
class Greeting(BaseMethod):
    """A custom greeting method that behaves like a static method."""
    
    def __call__(self, name):
        """Generate a static greeting for the given name."""
        # Even though we don't bind to new instances, we can still access the instance
        return f"Hello, {name}! I'm {self.__self__.__class__.__name__}!"

# Example Class
class Example:
    pass

example = Example()

# Create a class with the static greeting method
class Greeter:
    # Add the method as a class attribute
    greet = Greeting(instance=example, is_binding=False)  # Set is_binding=False to prevent binding to new instances

# Create an instance
greeter = Greeter()

# Call the method through the instance
print(greeter.greet("World"))

# Call the method through the class
print(Greeter.greet("Universe"))

# Enable binding to demonstrate the difference
greeter.greet.is_binding = True
print(greeter.greet("Universe"))

Hello, World! I'm Example!
Hello, Universe! I'm Example!
Hello, Universe! I'm Greeter!


#### Direct Implementation with Attribute Binding

You can create methods that bind to instances and set themselves as attributes on the instance. This is useful for dynamically adding methods to instances.

In [36]:
# Create a custom method with direct implementation
class AreaCalculator(BaseMethod):
    """A method that calculates the area of a shape."""
    
    def __call__(self):
        """Calculate the area based on the shape's attributes."""
        # Access the bound instance through self.__self__
        instance = self.__self__
        if hasattr(instance, 'width') and hasattr(instance, 'height'):
            return instance.width * instance.height
        elif hasattr(instance, 'radius'):
            import math
            return math.pi * instance.radius ** 2
        else:
            return "Unknown shape type"

# Create a rectangle class
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

# Create a circle class
class Circle:
    def __init__(self, radius):
        self.radius = radius

# Create instances
rect = Rectangle(5, 3)
circle = Circle(4)

# Create an AreaCalculator and bind it to the instances
area_calculator = AreaCalculator()
area_calculator.bind_to_attribute(rect, Rectangle, "area")
area_calculator.bind_to_attribute(circle, Circle, "area")

# Call the methods through the instances
print(f"Rectangle area: {rect.area()}")
print(f"Circle area: {circle.area()}")

Rectangle area: 50.26548245743669
Circle area: 50.26548245743669


## Secondary Usage - Method Wrapping (continued)

Let's continue exploring the secondary usage of BaseMethod for wrapping existing methods.

### Controlling Binding Behavior with Wrapped Methods

When wrapping existing methods, you can control binding behavior through the `is_binding` flag. If `is_binding` is set to `False`, the method doesn't bind to new instances when binding is called.

In [37]:
# Define a method
def greet(self, name):
    """A static greeting function."""
    return f"Hello, {name}! I'm {self.__class__.__name__}!"

# Define a class
class Example:
    pass

# Create an instance
example = Example()

# Create a BaseMethod with is_binding=False
static_method = BaseMethod(greet, instance=example, is_binding=False)

# Create a class with the static method
class Greeter:
    greet = static_method  # It added to the class namespace and uses its descriptor methods

# Create an instance
greeter = Greeter()

# Call the method through the instance
print(greeter.greet("World"))

# Call the method through the class
print(Greeter.greet("Universe"))

# Enable binding to demonstrate binding
static_method.is_binding = True
print(greeter.greet("Universe"))

Hello, World! I'm Example!
Hello, Universe! I'm Example!
Hello, Universe! I'm Greeter!


### Binding Wrapped Methods to Attributes

When wrapping existing methods, you can bind them to instances and set them as attributes using the `bind_to_attribute` method. This is useful for dynamically adding methods to instances.

In [38]:
# Define a function to wrap
def calculate_area(self):
    """Calculate the area of a rectangle."""
    return self.width * self.height

# Create a class
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

# Create a Rectangle instance
rect = Rectangle(5, 3)

# Create a BaseMethod and bind it to the instance as an attribute
area_method = BaseMethod(calculate_area)
area_method.bind_to_attribute(rect, Rectangle, "area")

# Call the method through the instance
print(f"Rectangle area: {rect.area()}")

# Create another instance and bind the method to it
rect2 = Rectangle(10, 4)
area_method.bind_to_attribute(rect2, Rectangle, "area")

# Call the method through the second instance
print(f"Rectangle 2 area: {rect2.area()}")

Rectangle area: 15
Rectangle 2 area: 40


## Module Interaction

The `BaseMethod` module interacts with other modules in the baseobjects package, particularly `BaseCallable` and `BaseReducible`. These interactions provide enhanced functionality for method-like objects.

### Interaction with BaseCallable

`BaseMethod` inherits from `BaseCallable`, which means it also inherits all the functionality of `BaseCallable`, including function wrapping, coroutine support, and attribute preservation:

In [39]:
# Define a class with a method
class Calculator:
    def __init__(self, name):
        self.name = name

    def add(self, x, y):
        """Add two numbers."""
        return x + y

    def subtract(self, x, y):
        """Subtract y from x."""
        return x - y

# Create BaseMethod instances that wrap the methods
add_method = BaseMethod(Calculator.add)
subtract_method = BaseMethod(Calculator.subtract)

# Create a Calculator instance
calc = Calculator("MyCalc")

# Bind the methods to the instance
bound_add = add_method.__get__(calc, Calculator)
bound_subtract = subtract_method.__get__(calc, Calculator)

# Call the bound methods
print(f"{calc.name} adds 5 + 3 = {bound_add(5, 3)}")
print(f"{calc.name} subtracts 5 - 3 = {bound_subtract(5, 3)}")

# Check if the docstrings are preserved
print(f"add_method docstring: {add_method.__doc__}")
print(f"subtract_method docstring: {subtract_method.__doc__}")

MyCalc adds 5 + 3 = 8
MyCalc subtracts 5 - 3 = 2
add_method docstring: Add two numbers.
subtract_method docstring: Subtract y from x.


### Interaction with Pickling

`BaseMethod` properly handles pickling and unpickling, even when bound to instances. It converts the weak reference to the bound instance into a strong reference for pickling, then back to a weak reference after unpickling:

In [40]:
import pickle

# Define a class with a method
class Counter:
    def __init__(self, initial=0):
        self.count = initial

    def increment(self, amount=1):
        """Increment the counter by the given amount."""
        self.count += amount
        return self.count

# Create a Counter instance
counter = Counter(10)

# Create a BaseMethod that wraps the increment method and binds it to the counter
increment_method = BaseMethod(Counter.increment, instance=counter, owner=Counter)

# Call the method
print(f"Counter before: {counter.count}")
print(f"Increment result: {increment_method(5)}")
print(f"Counter after: {counter.count}")

# Pickle the method
item = (increment_method, counter)  # increment_method has a weak reference to the counter, so a strong reference to the counter must be pickled as well
pickled_item  = pickle.dumps(item)
print(f"Pickled data (bytes): {pickled_item[:30]}... (truncated)")

# Unpickle the method
unpickled_method, unpickled_counter = pickle.loads(pickled_item)

# Call the unpickled method
print(f"Counter before: {counter.count}")
print(f"Unpickled increment result: {unpickled_method(5)}")
print(f"Original Counter (should not have changed): {counter.count}")

Counter before: 10
Increment result: 15
Counter after: 15
Pickled data (bytes): b'\x80\x04\x95V\x01\x00\x00\x00\x00\x00\x00\x8c\x1ebaseobjects.bases'... (truncated)
Counter before: 15
Unpickled increment result: 20
Original Counter (should not have changed): 15


## Advanced Features

The `BaseMethod` class provides several advanced features that make it powerful for creating custom method-like objects.

### Creating a Custom Method Class

You can create your own custom method class by inheriting from `BaseMethod` and overriding its methods:

In [41]:
class LoggingMethod(BaseMethod):
    """A method that logs its calls."""

    def __init__(self, func=None, instance=None, owner=None, log_prefix="METHOD", *args, **kwargs):
        super().__init__(func, instance, owner, *args, **kwargs)
        self.log_prefix = log_prefix
        self.call_count = 0

    def __call__(self, *args, **kwargs):
        self.call_count += 1
        instance = self.__self__
        instance_name = getattr(instance, 'name', str(instance))
        print(f"{self.log_prefix} #{self.call_count}: {instance_name}.{self.__wrapped__.__name__}({args}, {kwargs})")
        result = super().__call__(*args, **kwargs)
        print(f"{self.log_prefix} #{self.call_count} result: {result}")
        return result

# Define a class with methods
class MathOperations:
    def __init__(self, name):
        self.name = name

    def multiply(self, x, y):
        """Multiply two numbers."""
        return x * y

    def divide(self, x, y):
        """Divide x by y."""
        return x / y

# Create a MathOperations instance
math_ops = MathOperations("MathOps")

# Create LoggingMethod instances that wrap the methods and bind them to the instance
logging_multiply = LoggingMethod(MathOperations.multiply, instance=math_ops, owner=MathOperations, log_prefix="MATH")
logging_divide = LoggingMethod(MathOperations.divide, instance=math_ops, owner=MathOperations, log_prefix="MATH")

# Call the methods
result1 = logging_multiply(5, 3)
result2 = logging_divide(10, 2)

print(f"Total calls: {logging_multiply.call_count + logging_divide.call_count}")

MATH #1: MathOps.multiply((5, 3), {})
MATH #1 result: 15
MATH #1: MathOps.divide((10, 2), {})
MATH #1 result: 5.0
Total calls: 2


### Method Replacement

`BaseMethod` can be used to replace methods on existing instances:

In [42]:
class OriginalClass:
    def __init__(self, name):
        self.name = name

    def method(self, arg):
        """Original method."""
        return f"Original {self.name}.method({arg})"

# Create an instance
obj = OriginalClass("MyObject")

# Call the original method
print(obj.method("test"))

# Define a replacement method
def replacement_method(self, arg):
    """Replacement method."""
    return f"Replaced {self.name}.method({arg})"

# Create a BaseMethod that wraps the replacement method and bind it to the instance
replacement = BaseMethod(replacement_method)
replacement.bind_to_attribute(obj, OriginalClass, "method")

# Call the replaced method
print(obj.method("test"))

# Check if the docstring is updated
print(f"New docstring: {obj.method.__doc__}")

Original MyObject.method(test)
Replaced MyObject.method(test)
New docstring: Replacement method.


## Examples

Let's explore some practical examples of using the `BaseMethod` module.

### Creating a Tracing Method Decorator

We can use `BaseMethod` to create a decorator that adds tracing to methods:

In [43]:
import time

class TracingMethod(BaseMethod):
    """A method decorator that traces method calls with timing information."""

    def __call__(self, *args, **kwargs):
        instance = self.__self__
        class_name = instance.__class__.__name__
        method_name = self.__wrapped__.__name__

        print(f"Entering {class_name}.{method_name}({args}, {kwargs})")
        start_time = time.time()

        try:
            result = super().__call__(*args, **kwargs)
            print(f"Exiting {class_name}.{method_name} (time: {time.time() - start_time:.6f}s)")
            return result
        except Exception as e:
            print(f"Exception in {class_name}.{method_name}: {type(e).__name__}: {e}")
            raise

# Create a tracing method decorator
def trace(func):
    return TracingMethod(func)

# Define a class with traced methods
class Database:
    def __init__(self, name):
        self.name = name
        self.connected = False

    @trace
    def connect(self):
        """Connect to the database."""
        print(f"Connecting to {self.name}...")
        time.sleep(0.5)  # Simulate connection delay
        self.connected = True
        return True

    @trace
    def query(self, sql):
        """Execute a query on the database."""
        if not self.connected:
            raise RuntimeError("Not connected to the database")
        print(f"Executing query: {sql}")
        time.sleep(0.3)  # Simulate query execution
        return f"Result of {sql}"

    @trace
    def disconnect(self):
        """Disconnect from the database."""
        print(f"Disconnecting from {self.name}...")
        time.sleep(0.2)  # Simulate disconnection delay
        self.connected = False
        return True

# Create a Database instance
db = Database("MyDB")

# Use the traced methods
db.connect()
result = db.query("SELECT * FROM users")
print(f"Query result: {result}")
db.disconnect()

# Try to query after disconnecting
try:
    db.query("SELECT * FROM products")
except RuntimeError as e:
    print(f"Expected error: {e}")

Entering Database.connect((), {})
Connecting to MyDB...
Exiting Database.connect (time: 0.501007s)
Entering Database.query(('SELECT * FROM users',), {})
Executing query: SELECT * FROM users
Exiting Database.query (time: 0.300905s)
Query result: Result of SELECT * FROM users
Entering Database.disconnect((), {})
Disconnecting from MyDB...
Exiting Database.disconnect (time: 0.200412s)
Entering Database.query(('SELECT * FROM products',), {})
Exception in Database.query: RuntimeError: Not connected to the database
Expected error: Not connected to the database


### Creating a Caching Math Calculator

We can use `BaseMethod` to create method-like objects that directly implement caching functionality:

In [44]:
class CachingMathMethod(BaseMethod):
    """A method that directly implements math calculations with caching."""
    
    def __init__(self, operation_type, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.operation_type = operation_type
        self.cache = {}
        self.computation_count = 0
    
    def __call__(self, n):
        """Perform a math calculation with caching."""
        # Create a key from the arguments
        key = n
        
        # Check if the result is already in the cache
        if key in self.cache:
            print(f"Cache hit for {self.operation_type}({n})")
            return self.cache[key]
        
        print(f"Cache miss for {self.operation_type}({n})")
        
        # Directly implement the calculation based on operation type
        if self.operation_type == "factorial":
            self.computation_count += 1
            if n <= 1:
                result = 1
            else:
                # Recursive call to self for factorial
                result = n * self(n - 1)
                
        elif self.operation_type == "fibonacci":
            self.computation_count += 1
            if n <= 1:
                result = n
            else:
                # Recursive calls to self for fibonacci
                result = self(n - 1) + self(n - 2)
                
        else:
            raise ValueError(f"Unknown operation type: {self.operation_type}")
        
        # Store the result in the cache
        self.cache[key] = result
        return result
    
    def clear_cache(self):
        """Clear the cache."""
        self.cache = {}
    
    def get_cache_size(self):
        """Get the number of cached results."""
        return len(self.cache)

# Define a calculator class that uses our caching methods
class MathCalculator:
    def __init__(self, name):
        self.name = name
        
        # Create method instances with direct implementation
        self.factorial = CachingMathMethod("factorial", instance=self)
        self.fibonacci = CachingMathMethod("fibonacci", instance=self)
    
    def get_total_computations(self):
        """Get the total number of computations performed."""
        return self.factorial.computation_count + self.fibonacci.computation_count
    
    def get_cache_stats(self):
        """Get statistics about the cache."""
        return {
            "factorial_cache_size": self.factorial.get_cache_size(),
            "fibonacci_cache_size": self.fibonacci.get_cache_size(),
            "total_cached_results": self.factorial.get_cache_size() + self.fibonacci.get_cache_size()
        }

# Create a MathCalculator instance
calculator = MathCalculator("MathCalc")

# Calculate some factorials
print(f"factorial(5) = {calculator.factorial(5)}")
print(f"factorial(5) = {calculator.factorial(5)}")  # Should be cached
print(f"factorial(6) = {calculator.factorial(6)}")

# Calculate some Fibonacci numbers
print(f"fibonacci(10) = {calculator.fibonacci(10)}")
print(f"fibonacci(10) = {calculator.fibonacci(10)}")  # Should be cached
print(f"fibonacci(11) = {calculator.fibonacci(11)}")

# Print statistics
print(f"Total computations: {calculator.get_total_computations()}")
print(f"Cache statistics: {calculator.get_cache_stats()}")

Cache miss for factorial(5)
Cache miss for factorial(4)
Cache miss for factorial(3)
Cache miss for factorial(2)
Cache miss for factorial(1)
factorial(5) = 120
Cache hit for factorial(5)
factorial(5) = 120
Cache miss for factorial(6)
Cache hit for factorial(5)
factorial(6) = 720
Cache miss for fibonacci(10)
Cache miss for fibonacci(9)
Cache miss for fibonacci(8)
Cache miss for fibonacci(7)
Cache miss for fibonacci(6)
Cache miss for fibonacci(5)
Cache miss for fibonacci(4)
Cache miss for fibonacci(3)
Cache miss for fibonacci(2)
Cache miss for fibonacci(1)
Cache miss for fibonacci(0)
Cache hit for fibonacci(1)
Cache hit for fibonacci(2)
Cache hit for fibonacci(3)
Cache hit for fibonacci(4)
Cache hit for fibonacci(5)
Cache hit for fibonacci(6)
Cache hit for fibonacci(7)
Cache hit for fibonacci(8)
fibonacci(10) = 55
Cache hit for fibonacci(10)
fibonacci(10) = 55
Cache miss for fibonacci(11)
Cache hit for fibonacci(10)
Cache hit for fibonacci(9)
fibonacci(11) = 89
Total computations: 18
Cach

### Creating a Form Validator

We can use `BaseMethod` to create method-like objects that directly implement validation functionality:

In [45]:
class FormFieldValidator(BaseMethod):
    """A method that directly implements form field validation."""
    
    def __init__(self, field_name, validation_rules, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.field_name = field_name
        self.validation_rules = validation_rules
        self.validation_history = []
    
    def __call__(self, value):
        """Validate a form field value against the rules."""
        # Access the bound instance (form) through self.__self__
        form = self.__self__
        
        # Track validation attempt
        validation_record = {
            "field": self.field_name,
            "value": value,
            "valid": True,
            "errors": []
        }
        
        # Apply each validation rule
        for rule_name, rule_func in self.validation_rules.items():
            try:
                if not rule_func(value):
                    error_msg = f"Failed validation rule '{rule_name}'"
                    validation_record["valid"] = False
                    validation_record["errors"].append(error_msg)
                    raise ValueError(f"Invalid {self.field_name}: {error_msg}")
            except Exception as e:
                if not isinstance(e, ValueError):  # Only add if not our own error
                    validation_record["valid"] = False
                    validation_record["errors"].append(str(e))
                    raise ValueError(f"Invalid {self.field_name}: {str(e)}")
        
        # Store validation result in history
        self.validation_history.append(validation_record)
        
        # If we got here, validation passed
        print(f"Validation passed for {self.field_name}: {value}")
        
        # Store the validated value in the form
        form.validated_data[self.field_name] = value
        return value
    
    def get_validation_stats(self):
        """Get statistics about validations performed."""
        total = len(self.validation_history)
        successful = sum(1 for record in self.validation_history if record["valid"])
        failed = total - successful
        
        return {
            "total_validations": total,
            "successful_validations": successful,
            "failed_validations": failed,
            "success_rate": successful / total if total > 0 else 0
        }

# Define a form class that uses our validator methods
class UserRegistrationForm:
    def __init__(self):
        self.validated_data = {}
        
        # Create validator methods with direct implementation
        self.validate_username = FormFieldValidator("username", {
            "type": lambda x: isinstance(x, str),
            "length": lambda x: 3 <= len(x) <= 20,
            "alphanumeric": lambda x: x.isalnum()
        }, instance=self)
        
        self.validate_email = FormFieldValidator("email", {
            "type": lambda x: isinstance(x, str),
            "format": lambda x: '@' in x and '.' in x.split('@')[1]
        }, instance=self)
        
        self.validate_age = FormFieldValidator("age", {
            "type": lambda x: isinstance(x, int),
            "range": lambda x: x >= 18
        }, instance=self)
    
    def submit(self):
        """Submit the form if all required fields are validated."""
        required_fields = ["username", "email"]
        for field in required_fields:
            if field not in self.validated_data:
                raise ValueError(f"Missing required field: {field}")
        
        return f"Form submitted successfully with data: {self.validated_data}"

# Create a form instance
form = UserRegistrationForm()

# Validate fields with valid values
try:
    form.validate_username("johndoe")
    form.validate_email("john@example.com")
    form.validate_age(25)
    
    # Submit the form
    result = form.submit()
    print(result)
except ValueError as e:
    print(f"Validation error: {e}")

# Try validating with invalid values
try:
    form.validate_username("jo")  # Too short
except ValueError as e:
    print(f"Expected error: {e}")

try:
    form.validate_email("invalid-email")  # Missing @ and domain
except ValueError as e:
    print(f"Expected error: {e}")

try:
    form.validate_age(16)  # Too young
except ValueError as e:
    print(f"Expected error: {e}")

# Print validation statistics
print("\nValidation statistics:")
print(f"Username: {form.validate_username.get_validation_stats()}")
print(f"Email: {form.validate_email.get_validation_stats()}")
print(f"Age: {form.validate_age.get_validation_stats()}")

Validation passed for username: johndoe
Validation passed for email: john@example.com
Validation passed for age: 25
Form submitted successfully with data: {'username': 'johndoe', 'email': 'john@example.com', 'age': 25}
Validation passed for username: jo
Validation passed for email: invalid-email
Validation passed for age: 16

Validation statistics:
Username: {'total_validations': 2, 'successful_validations': 1, 'failed_validations': 1, 'success_rate': 0.5}
Email: {'total_validations': 2, 'successful_validations': 1, 'failed_validations': 1, 'success_rate': 0.5}
Age: {'total_validations': 2, 'successful_validations': 1, 'failed_validations': 1, 'success_rate': 0.5}


## API Highlights

Here are the key components of the `BaseMethod` module API:

### BaseMethod
- `__init__(func=None, instance=None, owner=None, *args, is_binding=True, init=True, **kwargs)`: Initialize a new BaseMethod instance
- `__self__`: Property to get/set the object to which this method is bound
- `is_binding`: Determines if this callable will bind to another object when accessed as an attribute
- `bind_self(instance=None, owner=None)`: Binds this method to an instance and/or owner class
- `bind_to_attribute(instance=None, owner=None, name=None)`: Binds this method to an instance and sets it as an attribute on that instance
- `call_binding(*args, **kwargs)`: Binds the wrapped function to the stored instance and calls it

For more detailed information, consult the full API documentation.

## Troubleshooting / FAQs

### Q: What's the difference between BaseMethod and a regular Python method?

A: `BaseMethod` is a more flexible and customizable version of a regular Python method. It allows you to:
1. Create method-like objects that can be bound to instances
2. Control binding behavior through the `is_binding` flag
3. Bind methods to instances dynamically
4. Create custom method types by subclassing
5. Preserve function attributes and handle pickling properly

### Q: How does BaseMethod handle weak references?

A: `BaseMethod` stores a weak reference to the bound instance using Python's `weakref` module. This prevents memory leaks that could occur if the method held a strong reference to the instance. When the method is called, it retrieves the bound instance from the weak reference and passes it as the first argument to the wrapped function.

### Q: Can I use BaseMethod with coroutines?

A: Yes, `BaseMethod` inherits from `BaseCallable`, which properly handles coroutines. You can wrap async functions with `BaseMethod` and they will maintain their async behavior. When calling a `BaseMethod` that wraps a coroutine, you need to use `await` to get the result.

## Conclusion and Next Steps

In this tutorial, we've explored the `BaseMethod` module and its primary class, `BaseMethod`. We've seen how this class extends `BaseCallable` to create callable objects that behave like methods.

**The primary purpose of BaseMethod is to directly implement functionality in its own methods** through subclassing and overriding `__call__`. This approach provides better performance, more flexibility, and cleaner code when creating custom method-like objects. By subclassing BaseMethod, you can create powerful method-like objects that maintain their own state, implement custom behavior, and still benefit from the method binding semantics provided by BaseMethod.

While BaseMethod can also wrap existing methods, this is a secondary purpose that doesn't fully utilize the power of the class. Direct implementation through subclassing is the recommended approach for most use cases.

BaseMethod handles edge cases like proper pickling, weak references, and binding behavior that are often overlooked in custom method implementations, making it a robust foundation for creating custom method types.

> **Note:** For decorator functionality, use the `BaseDecorator` class from the `baseobjects.functions` package instead. `BaseDecorator` is specifically designed for creating decorators with extended functionality.

### Next Steps

- Try creating your own custom method classes by extending `BaseMethod` and directly implementing functionality
- Explore the `BaseFunction` module, which extends `BaseCallable` to create function-like objects with direct implementation
- Check out the examples in the baseobjects package that demonstrate more advanced uses of `BaseMethod`
- Check out the `BaseDecorator` module in the `baseobjects.functions` package for creating decorators
- Consult the full API documentation for more detailed information on the `BaseMethod` module